# Klassifikation mit allen Werten

`JobSat` als Zielwert


In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC


## Daten laden

Im One-Hot-Encoded Datensatz sind die One-Hot-Encoded Spalten mit bool Werten. Um es etwas einfacher zu gestalten werden diese Werte hier in Integer-Werte (False -> 0; True -> 1) umgewandelt.

In [ ]:

df = pd.read_csv("survey_results_cleaned_final.csv")


bool_cols = df.select_dtypes(include=['bool']).columns

for col in bool_cols:
    df[col] = df[col].astype(int)
df.dtypes

## Zielvariable in Klassen einteilen 

Dadurch haben wir mehr Trainingsdaten, als wenn wir `JobSat` von 0-10 als Klassen defonieren würden

Klassen
- **Low**: 0–3
- **Medium**: 4–6
- **High**: 7–10


In [ ]:
# Nur Zeilen behalten, wo JobSat vorhanden ist
df = df.dropna(subset=["JobSat"]).copy()

def map_jobsat(x):
    x = float(x)
    if x <= 3:
        return "Low"
    elif x <= 6:
        return "Medium"
    else:
        return "High"


## Feature-Spalten bestimmen

- Textspalten: `object` (Strings)
- Numerische Spalten: `int/float`



In [ ]:
# Textspalten (Strings)
text_cols = df.select_dtypes(include=["object"]).columns.tolist()

df["__text__"] = df[text_cols].fillna("").agg(" ".join, axis=1)

# Numerische Spalten
num_cols = df.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()

# Zielspalte aus numerischen Features entfernen (falls vorhanden)
num_cols = [c for c in num_cols if c != "JobSat"] #?

df = df.dropna(subset=["__text__"] + num_cols + ["JobSat"])

y = df["JobSat"].apply(map_jobsat)

print("Textspalten:", text_cols)
print("Numerische Spalten:", num_cols)

# Feature-Matrix aus den ausgewählten Spalten
X = df[["__text__"] + num_cols].copy()


X.head()

## Train/Test Split



In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))
print("Train class distribution:\n", y_train.value_counts(normalize=True))
print("Test class distribution:\n", y_test.value_counts(normalize=True))

## Preprocessing für Text und numerische Spalten




In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("text", TfidfVectorizer(), "__text__"),
        ("num", StandardScaler(), num_cols)
    ],
    remainder="drop"
)


## Pipeline definieren

- Preprocessing
- Feature-Selektion (SelectFromModel mit L1-LinearSVC)
- Klassifikator (LinearSVC)


In [ ]:
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("feature_selection", SelectFromModel(LinearSVC(penalty="l1", dual=False, C=0.5))),
    ("classifier", LinearSVC())
])



## GridSearchCV



In [ ]:

parameters = {
    # TF-IDF: nur word, nur die zwei wichtigsten Varianten
    "preprocessing__text__analyzer": ["word"],
    "preprocessing__text__ngram_range": [(1, 1), (1, 2)],
    "preprocessing__text__max_df": [0.9],
    "preprocessing__text__min_df": [2],

    # Klassifikator: 2 sinnvolle Regularisierungen + optional balancing
    "classifier__C": [1.0, 2.0],
    "classifier__class_weight": [None, "balanced"],
}

grid = GridSearchCV(pipeline, param_grid=parameters, verbose=2, cv=3, n_jobs=-1)


## Grid Search + Beste Parameter


In [ ]:

grid.fit(X_train, y_train)

print("Beste Performance:", grid.best_score_)
print("Beste Parameter:\n", grid.best_params_)

## Evaluation auf Testdaten


In [ ]:
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print("Classification Report (Test):")
print(classification_report(y_test, y_pred))

print("Confusion Matrix (rows=true, cols=pred):")
print(confusion_matrix(y_test, y_pred, labels=["Low", "Medium", "High"]))

In [ ]:
y_pred_train = best_model.predict(X_train)

print("Classification Report (Train):")
print(classification_report(y_train, y_pred_train))